In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image, ImageDraw, ImageFilter, ImageFont
import colorsys
import random
import math
from pathlib import Path

In [ ]:
# ============================================================
# КОНФИГИ
# ============================================================
TRAIN_DIR = '/kaggle/input/competitions/dl-lab-4-ocr/train/train'
TEST_DIR  = '/kaggle/input/competitions/dl-lab-4-ocr/test/test'
TRAIN_CSV = '/kaggle/input/competitions/dl-lab-4-ocr/train.csv'

random.seed(42)
np.random.seed(42)

In [ ]:
# ============================================================
# УТИЛИТЫ — загрузка реальных изображений
# ============================================================
def load_image_rgb(path: str) -> np.ndarray:
    img = cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_COLOR)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

In [ ]:
# ============================================================
# ГЕНЕРАТОР ФОНА
# Цвет: от жёлтого до белого (HSV: H∈[30,75]°, S∈[0,1], V∈[0.75,1])
# + случайный гауссов шум + размытие
# ============================================================
def generate_background(width: int, height: int) -> Image.Image:
    """
    Генерирует фон: жёлтый / серый / белый — случайный.
    Добавляет гауссов шум и лёгкое размытие на БОЛЬШОМ холсте,
    затем ресемплит до (width, height).
    """
    # --- выбор цвета ---
    bg_type = random.choice(['yellow', 'gray', 'white'])
    if bg_type == 'yellow':
        h = random.uniform(30, 75) / 360.0
        s = random.uniform(0.2, 1.0)
        v = random.uniform(0.75, 1.0)
    elif bg_type == 'gray':
        h = random.uniform(0, 1)
        s = random.uniform(0.0, 0.15)
        v = random.uniform(0.55, 0.92)
    else:  # white
        h = random.uniform(0, 1)
        s = random.uniform(0.0, 0.08)
        v = random.uniform(0.90, 1.0)

    r, g, b = colorsys.hsv_to_rgb(h, s, v)
    base_color = (int(r * 255), int(g * 255), int(b * 255))

    # --- рисуем на большом холсте для антиалиасинга ---
    SCALE = 4
    big_w, big_h = width * SCALE, height * SCALE
    arr = np.full((big_h, big_w, 3), base_color, dtype=np.uint8)

    # --- шум ---
    noise_sigma = random.uniform(0, 30)
    if noise_sigma > 0:
        noise = np.random.normal(0, noise_sigma, arr.shape).astype(np.int16)
        arr = np.clip(arr.astype(np.int16) + noise, 0, 255).astype(np.uint8)

    img = Image.fromarray(arr)

    # --- размытие ---
    if random.random() < 0.6:
        radius = random.uniform(0, 2.5)
        img = img.filter(ImageFilter.GaussianBlur(radius=radius))

    # --- даунсемплинг до целевого размера ---
    img = img.resize((width, height), Image.LANCZOS)
    return img

In [ ]:
# ============================================================
# РИСОВАНИЕ ЦИФР — кастомный минималистичный стиль
#
# Описание шрифта:
#   1  — просто вертикальная черта (без засечки снизу)
#   6  — окружность (низ) + наклонная прямая вверх-вправо (из верха окружности)
#   9  — окружность (верх) + наклонная прямая вниз-вправо (из низа окружности)
#   остальные — классический минималистичный bold
# Рисуется на большом холсте, потом даунсемплируется.
# ============================================================

def _draw_digit(draw: ImageDraw.Draw,
                digit: str,
                x0: int, y0: int,
                cell_w: int, cell_h: int,
                sw: int,
                color=(0, 0, 0)):
    """
    Рисует одну цифру digit в прямоугольнике (x0,y0)—(x0+cell_w, y0+cell_h).
    sw — толщина обводки (stroke width).
    Все примитивы через draw.line / draw.ellipse / draw.arc с width=sw.
    """
    # Рабочая область с отступами
    pad = sw
    x1, y1 = x0 + pad, y0 + pad
    x2, y2 = x0 + cell_w - pad, y0 + cell_h - pad
    cx = (x1 + x2) // 2          # центр по X
    cy = (y1 + y2) // 2          # центр по Y
    W  = x2 - x1                  # рабочая ширина
    H  = y2 - y1                  # рабочая высота
    sw2 = sw                       # алиас

    def line(*pts, w=sw2):
        draw.line(pts, fill=color, width=w)

    def arc(bbox, start, end, w=sw2):
        draw.arc(bbox, start=start, end=end, fill=color, width=w)

    def ellipse(bbox, w=sw2):
        draw.ellipse(bbox, outline=color, width=w)

    if digit == '0':
        ellipse([x1, y1, x2, y2])

    elif digit == '1':
        # Вертикальная черта по центру — без засечек
        line((cx, y1), (cx, y2))
        # Маленькая диагональ сверху-слева
        line((cx - W // 4, y1 + H // 5), (cx, y1))

    elif digit == '2':
        # Верхняя дуга
        arc([x1, y1, x2, y1 + H // 2], start=200, end=360)
        # Диагональ вниз-влево
        line((x2, y1 + H // 4), (x1, y2))
        # Нижняя горизонталь
        line((x1, y2), (x2, y2))

    elif digit == '3':
        # Верхняя дуга
        arc([x1, y1, x2, cy + sw], start=230, end=360 + 130)
        # Нижняя дуга
        arc([x1, cy - sw, x2, y2], start=230, end=360 + 130)

    elif digit == '4':
        # Левая наклонная черта
        line((x1, y1), (x1, cy))
        # Горизонталь
        line((x1, cy), (x2, cy))
        # Правая вертикаль
        line((x2, y1), (x2, y2))

    elif digit == '5':
        # Верхняя горизонталь
        line((x1, y1), (x2, y1))
        # Левая вертикаль (верхняя половина)
        line((x1, y1), (x1, cy))
        # Средняя горизонталь
        line((x1, cy), (x2 - W // 5, cy))
        # Правая нижняя дуга
        arc([x1, cy, x2, y2], start=270, end=270 + 300)

    elif digit == '6':
        # Окружность (нижняя часть — основная петля)
        r = H // 3
        circ_cx = cx
        circ_cy = y2 - r
        ellipse([circ_cx - r, circ_cy - r, circ_cx + r, circ_cy + r])
        # Наклонная прямая вверх-вправо из верхней точки окружности
        top_x = circ_cx
        top_y = circ_cy - r
        line((top_x, top_y), (x2, y1))

    elif digit == '7':
        # Верхняя горизонталь
        line((x1, y1), (x2, y1))
        # Диагональ вниз-влево
        line((x2, y1), (x1 + W // 4, y2))
        # Маленькая горизонталь посередине
        mid_x = (x1 + W // 4 + x2) // 2
        mid_y = (y1 + y2) // 2
        line((mid_x - W // 5, mid_y), (mid_x + W // 5, mid_y), w=max(1, sw - 1))

    elif digit == '8':
        r_top = H // 4
        r_bot = H // 3
        top_cy = y1 + r_top
        bot_cy = y2 - r_bot
        ellipse([cx - r_top, top_cy - r_top, cx + r_top, top_cy + r_top])
        ellipse([cx - r_bot, bot_cy - r_bot, cx + r_bot, bot_cy + r_bot])

    elif digit == '9':
        # Окружность (верхняя часть — основная петля)
        r = H // 3
        circ_cx = cx
        circ_cy = y1 + r
        ellipse([circ_cx - r, circ_cy - r, circ_cx + r, circ_cy + r])
        # Наклонная прямая вниз-вправо из нижней точки окружности
        bot_x = circ_cx
        bot_y = circ_cy + r
        line((bot_x, bot_y), (x2, y2))


def render_number_pil(number: str,
                      target_w: int,
                      target_h: int) -> Image.Image:
    """
    Рисует число number на прозрачном фоне (RGBA),
    число занимает ~95-100% высоты холста.
    Возвращает PIL Image (RGBA) размером target_w × target_h.
    """
    SCALE = 8  # рисуем в 8× для качественного антиалиасинга

    big_w = target_w * SCALE
    big_h = target_h * SCALE

    n_digits = len(number)
    # ширина ячейки для каждой цифры
    cell_w = big_w // n_digits
    cell_h = big_h
    sw = max(2, int(cell_h * 0.14))   # толщина ~14% высоты

    img = Image.new('RGBA', (big_w, big_h), (255, 255, 255, 0))
    draw = ImageDraw.Draw(img)

    for i, ch in enumerate(number):
        _draw_digit(draw, ch,
                    x0=i * cell_w, y0=0,
                    cell_w=cell_w, cell_h=cell_h,
                    sw=sw,
                    color=(0, 0, 0, 255))

    # даунсемплинг
    img = img.resize((target_w, target_h), Image.LANCZOS)
    return img

In [ ]:
# ============================================================
# ГЕНЕРАЦИЯ СИНТЕТИЧЕСКОГО ИЗОБРАЖЕНИЯ
# ============================================================

def generate_synthetic(price: int = None,
                        target_w: int = None,
                        target_h: int = None) -> tuple[Image.Image, int]:
    """
    Генерирует одно синтетическое изображение ценника.
    Возвращает (PIL Image RGB, цена).
    """
    if price is None:
        # 2-, 3-, 4-значное число с реалистичным распределением
        n_digits = random.choices([2, 3, 4], weights=[0.25, 0.50, 0.25])[0]
        lo = 10 ** (n_digits - 1)
        hi = 10 ** n_digits - 1
        price = random.randint(lo, hi)

    if target_w is None:
        target_w = random.randint(40, 60)
    if target_h is None:
        target_h = random.randint(15, 35)

    # --- фон ---
    bg = generate_background(target_w, target_h)

    # --- цифры (рисуем на большом холсте) ---
    SCALE = 8
    big_w, big_h = target_w * SCALE, target_h * SCALE

    digits_img = render_number_pil(str(price), target_w * SCALE, target_h * SCALE)

    # --- наклон ±15° ---
    angle = random.uniform(-15, 15)
    digits_rot = digits_img.rotate(angle, expand=False,
                                   resample=Image.BICUBIC,
                                   fillcolor=(255, 255, 255, 0))

    # --- накладываем цифры на фон ---
    bg_big = bg.resize((big_w, big_h), Image.NEAREST).convert('RGBA')
    bg_big.alpha_composite(digits_rot)
    result = bg_big.convert('RGB')

    # --- даунсемплинг ---
    result = result.resize((target_w, target_h), Image.LANCZOS)

    # --- финальный шум ---
    arr = np.array(result, dtype=np.float32)
    noise_sigma = random.uniform(0, 8)
    if noise_sigma > 0:
        arr += np.random.normal(0, noise_sigma, arr.shape)
        arr = np.clip(arr, 0, 255).astype(np.uint8)
    else:
        arr = arr.astype(np.uint8)

    # --- финальное размытие ---
    result = Image.fromarray(arr)
    if random.random() < 0.5:
        r = random.uniform(0, 0.6)
        result = result.filter(ImageFilter.GaussianBlur(radius=r))

    return result, price

In [ ]:
# ============================================================
# 1. ПЕРВЫЕ 100 ИЗОБРАЖЕНИЙ ИЗ TRAIN
# ============================================================
print("=" * 60)
print("  1. Первые 100 изображений из тренировочного датасета")
print("=" * 60)

train_df = pd.read_csv(TRAIN_CSV, sep='\t')
# поддержка как пробела, так и табуляции как разделителя
if 'Price' not in train_df.columns:
    train_df = pd.read_csv(TRAIN_CSV, sep=r'\s+')

sample = train_df.head(100)

fig, axes = plt.subplots(10, 10, figsize=(20, 22))
fig.suptitle('TRAIN — первые 100 изображений (реальные)',
             fontsize=16, fontweight='bold', y=1.005)

for i, ax in enumerate(axes.flat):
    if i < len(sample):
        path = os.path.join(TRAIN_DIR, sample.iloc[i]['Filename'])
        img  = load_image_rgb(path)
        ax.imshow(img, aspect='auto')
        ax.set_title(str(sample.iloc[i]['Price']),
                     fontsize=9, fontweight='bold', color='darkred', pad=2)
    ax.axis('off')

plt.tight_layout()
plt.savefig('train_grid.png', dpi=100, bbox_inches='tight')
plt.show()
print("  Готово.\n")

In [ ]:
# ============================================================
# 2а. ВИЗУАЛИЗАЦИЯ ФОНОВ (20 штук, без цифр)
# ============================================================
print("=" * 60)
print("  2а. 20 сгенерированных фонов (без цифр)")
print("=" * 60)

fig, axes = plt.subplots(4, 5, figsize=(15, 13))
fig.suptitle('Сгенерированные фоны — 20 штук\n'
             '(жёлтый / серый / белый, случайное насыщение, шум, размытие)',
             fontsize=13, fontweight='bold')

random.seed(0); np.random.seed(0)

for i, ax in enumerate(axes.flat):
    w = random.randint(40, 60)
    h = random.randint(15, 35)
    bg = generate_background(w, h)
    # Показываем в увеличенном виде для наглядности
    bg_big = bg.resize((w * 8, h * 8), Image.NEAREST)
    ax.imshow(np.array(bg_big))
    ax.set_title(f'{w}×{h} px', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig('backgrounds.png', dpi=100, bbox_inches='tight')
plt.show()
print("  Готово.\n")

In [ ]:
# ============================================================
# 2б. ДЕМОНСТРАЦИЯ ШРИФТА: "1234567890"
# ============================================================
print("=" * 60)
print("  2б. Демонстрация кастомного шрифта: «1234567890»")
print("=" * 60)

DEMO_STR   = '1234567890'
DEMO_H_BIG = 400          # высота демонстрационного холста
DIGIT_W    = 90
TOTAL_W    = DIGIT_W * len(DEMO_STR)

demo_img = Image.new('RGB', (TOTAL_W, DEMO_H_BIG), (240, 240, 240))
draw     = ImageDraw.Draw(demo_img)

sw_demo = max(3, int(DEMO_H_BIG * 0.13))
for idx, ch in enumerate(DEMO_STR):
    _draw_digit(draw, ch,
                x0=idx * DIGIT_W, y0=0,
                cell_w=DIGIT_W, cell_h=DEMO_H_BIG,
                sw=sw_demo,
                color=(20, 20, 20))

fig, ax = plt.subplots(figsize=(14, 4))
ax.imshow(np.array(demo_img))
ax.set_title('Кастомный минималистичный шрифт: «1234567890»\n'
             '(1 — без засечки, 6 — круг+линия вверх, 9 — круг+линия вниз)',
             fontsize=12, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig('font_demo.png', dpi=100, bbox_inches='tight')
plt.show()
print("  Готово.\n")

In [ ]:
# ============================================================
# 3. 100 СИНТЕТИЧЕСКИХ ИЗОБРАЖЕНИЙ
# ============================================================
print("=" * 60)
print("  3. 100 синтетических изображений")
print("=" * 60)

random.seed(7); np.random.seed(7)

synth_imgs   = []
synth_prices = []

for _ in range(100):
    img, price = generate_synthetic()
    synth_imgs.append(img)
    synth_prices.append(price)

fig, axes = plt.subplots(10, 10, figsize=(22, 24))
fig.suptitle('Синтетические данные — 100 изображений\n'
             '(случайный фон, шум, наклон ±15°, даунсемплинг)',
             fontsize=14, fontweight='bold', y=1.005)

for i, ax in enumerate(axes.flat):
    img_arr = np.array(synth_imgs[i])
    # Показываем в увеличенном виде (×6) для наглядности
    h, w = img_arr.shape[:2]
    big  = Image.fromarray(img_arr).resize((w * 6, h * 6), Image.NEAREST)
    ax.imshow(np.array(big), aspect='auto')
    ax.set_title(str(synth_prices[i]),
                 fontsize=8, fontweight='bold', color='navy', pad=2)
    ax.axis('off')

plt.tight_layout()
plt.savefig('synthetic_grid.png', dpi=100, bbox_inches='tight')
plt.show()

# Статистика
prices_arr = np.array(synth_prices)
print(f"\n  Статистика сгенерированных цен (100 шт.):")
print(f"    Мин      : {prices_arr.min()}")
print(f"    Макс     : {prices_arr.max()}")
print(f"    Среднее  : {prices_arr.mean():.1f}")
print(f"    2-знач.  : {(prices_arr < 100).sum()}")
print(f"    3-знач.  : {((prices_arr >= 100) & (prices_arr < 1000)).sum()}")
print(f"    4-знач.  : {(prices_arr >= 1000).sum()}")
print("\n  Готово. Все изображения сохранены.")